In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
df_raw = pd.read_parquet("/home/tamer/projects/mlops-practitioner/data/raw/green_tripdata_2023-01.parquet")
df_raw.head()

In [ ]:
df_raw.info()

In [ ]:
df_raw.size

In [ ]:
df_raw.describe()

In [ ]:
df_raw["lpep_dropoff_datetime"][:5]

In [ ]:
df_raw["lpep_pickup_datetime"][:5]

In [ ]:
df_raw["lpep_dropoff_datetime"][1] - df_raw["lpep_pickup_datetime"][1]

In [ ]:
# calculate the duration in minutes
df_raw['duration'] = df_raw.lpep_dropoff_datetime - df_raw.lpep_pickup_datetime
df_raw["duration"][:5]

In [ ]:
# convert the calculated duration from seconds to minutes
df_raw.duration.dt.total_seconds() / 60

In [ ]:
df_raw.duration = df_raw.duration.dt.total_seconds() / 60


In [ ]:
df_raw.duration

In [ ]:
df_raw.duration.min()

In [ ]:
df_raw.duration.max()

In [ ]:
# Filter realistic trip durations (e.g., 1 minute to 60 minutes)
df = df_raw[(df_raw.duration >= 1) & (df_raw.duration <= 60)].copy()

In [ ]:
df_raw.size

In [ ]:
df_raw["PULocationID"]

In [ ]:
df_raw.describe().T

In [ ]:
df = df_raw.copy()


In [ ]:
df["duration_min"] = (df["lpep_dropoff_datetime"] - df["lpep_pickup_datetime"]).dt.total_seconds() / 60
print(df["duration_min"].describe(percentiles=[.01, .5, .99]).round(2).to_string())
print("\nnegative durations:", (df["duration_min"] < 0).sum())
print("longer than 3 hours:", (df["duration_min"] > 180).sum())

In [ ]:
df.isna().sum()

In [ ]:
df.columns

In [ ]:
drop_cols = ["store_and_fwd_flag", "RatecodeID","passenger_count","ehail_fee","payment_type","trip_type","congestion_surcharge"]
df = df.drop(columns=drop_cols)


In [ ]:
df.columns

In [ ]:
df[["duration","duration_min"]]

In [ ]:
df.head()

In [ ]:
dropped_cols = ["lpep_pickup_datetime","lpep_dropoff_datetime","PULocationID","DOLocationID"]
df.drop(dropped_cols, axis=1, inplace=True)

In [ ]:
df.drop("VendorID", axis=1, inplace=True) 

In [ ]:
df.columns

In [ ]:
df = df.drop_duplicates()

In [ ]:
# 7. rows without pickup/dropoff zone
df = df.dropna()

In [ ]:
df.shape

In [ ]:
df_raw.shape

In [ ]:
import pathlib as Path

In [ ]:
SEED = 42
ROOT = r"/home/tamer/projects/mlops-practitioner/notebooks/01_baseline.ipynb"
RAW_DATA = r"/home/tamer/projects/mlops-practitioner/data/raw/green_tripdata_2023-01.parquet"
CLEAN_FILE = r"/home/tamer/projects/mlops-practitioner/data/cleaned/green_tripdata_2023-01_clean.parquet"
PARAMS = {
    "seed": SEED,
    "min_duration_min": 1,      # shorter trips are meter glitches
    "max_duration_min": 60,     # longer trips are very rare
    "min_distance_mi": 0.1,
    "max_distance_mi": 30,
    "max_passengers": 6,
    "val_size": 0.15,
    "test_size": 0.15,
    "rare_category_min_freq": 30,
}

In [ ]:
print(CLEAN_FILE)

In [ ]:
df.to_parquet(CLEAN_FILE, index=False)

In [ ]:
df_model = pd.read_parquet(CLEAN_FILE)
df_model.head()

In [ ]:
df_model.dtypes

In [ ]:
X = df_model.drop(["duration", "duration_min"],axis=1)
X.head()

In [ ]:
y = df_model["duration"]
y

# SPlitting the DATA

In [ ]:
X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=PARAMS["test_size"], random_state=SEED)


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_rest, y_rest, test_size=PARAMS["val_size"] / (1 - PARAMS["test_size"]), random_state=SEED)

print(f"train={len(X_train):,}  val={len(X_val):,}  test={len(X_test):,}")

# Scalling

In [ ]:
from sklearn.preprocessing import MinMaxScaler
# Scale features (drop ID)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Reattach IDs with aligned indices
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)


X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test.columns)


# Training Random Forest 

In [ ]:
params = {
"max_depth":70,
"random_state":42,
"n_estimators":30,
"n_jobs":-1,
"verbose":1,
"criterion":'absolute_error'
}

In [ ]:
from sklearn.ensemble import RandomForestRegressor

regr = RandomForestRegressor(**params)
regr.fit(X_train_scaled_df, y_train)

In [ ]:
regr.get_metadata_routing()

In [ ]:
y_pred = regr.predict(X_test_scaled_df)
print(y_pred)

In [ ]:
mse = mean_squared_error(y_test, y_pred)
print(mse)

In [ ]:
from sklearn.metrics import root_mean_squared_error
rmse = root_mean_squared_error(y_test, y_pred)
print(rmse)

In [ ]:
from sklearn.metrics import (
    mean_squared_error, 
    root_mean_squared_error, 
    mean_absolute_error, 
    r2_score, 
    mean_absolute_percentage_error, 
    max_error
)

In [ ]:
mse = mean_squared_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred)
max_err = max_error(y_test, y_pred)

# MLFlow

In [ ]:
import mlflow
# the Experiment name
mlflow.set_experiment("RandomForestRegressor Baseline")
# connect the ml flow with the ui url 
mlflow.set_tracking_uri("http://127.0.0.1:5000") # We used DagsHub for making the tracking public

with mlflow.start_run():
    # Log the params of the model we used in training
    # 1. Log the model's parameters
    params = {
        "max_depth": 70,
        "random_state": 42,
        "n_estimators": 30,
        "n_jobs": -1,
        "verbose": 1,
        "criterion": "absolute_error"
    }
    mlflow.log_params(params)

    # Initialize and train the model using those params
    regr = RandomForestRegressor(**params)
    regr.fit(X_train_scaled_df, y_train)

    # Generate predictions for evaluation
    y_pred = regr.predict(X_test_scaled_df)

    # logging the metrices we used
    # first we want to calc the metrices 
    mse = mean_squared_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    max_err = max_error(y_test, y_pred)

    mlflow.log_metrics({
    "mse": mse,
    "rmse": rmse,
    "mae": mae,
    "r2_score": r2,
    "mape": mape,
    "max_error": max_err
    })

    mlflow.sklearn.log_model(
        regr,
        artifact_path="random_forest_model",

        skops_trusted_types=["sklearn.tree._tree.Tree"] # That tells mlflow that my model is secure to serialize it because mlflow not trust the pickle models
    )

# Try Multiple models

In [ ]:
import time
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor
import mlflow

In [ ]:
# mlflow.set_tracking_uri("http://127.0.0.1:5000/")
# mlflow.set_experiment("taxi_model_comparison")


In [ ]:
# 2. Define the models and their important parameters in dictionaries
# Format: (Model Name, Model Class, Parameter Dictionary)
models_to_try = [
    (
        "Linear_Regression", 
        LinearRegression, 
        {"fit_intercept": True}
    ),
    (
        "Decision_Tree", 
        DecisionTreeRegressor, 
        {"max_depth": 100, "min_samples_split": 5, "random_state": 42, "criterion": "absolute_error"}
    ),
    (
        "Random_Forest", 
        RandomForestRegressor, 
        {"max_depth": 50, "n_estimators": 30, "min_samples_split": 5, "random_state": 42, "n_jobs": -1}
    ),
    (
        "XGBoost", 
        XGBRegressor, 
        {"max_depth": 20, "n_estimators": 30, "learning_rate": 0.1, "random_state": 42, "n_jobs": -1}
    )
]

## Connect to dagshub 
#### for make my experiments puplic in the cloud so any one in my team be able to see my experiments

In [ ]:
import dagshub
dagshub.init(repo_owner='tamer-elkoT', repo_name='mlops-practitioner', mlflow=True)


In [ ]:


mlflow.set_tracking_uri("https://dagshub.com/tamer-elkoT/mlops-practitioner.mlflow")
mlflow.set_experiment("NYC Ride Duration")

# 3. Iterate through the models
for model_name, ModelClass, params in models_to_try:
    with mlflow.start_run(run_name=model_name):
        print(f"Training {model_name}...")
        
        # Log only the explicitly defined important parameters
        mlflow.log_params(params)
        
        # Initialize the model by unpacking the dictionary
        model = ModelClass(**params)
        
        # Track training time
        start_time = time.time()
        model.fit(X_train_scaled_df, y_train)
        training_time = time.time() - start_time
        
        # Generate predictions
        y_pred = model.predict(X_test_scaled_df)
        
        # Calculate metrics
        rmse = root_mean_squared_error(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        
        # Log metrics
        mlflow.log_metrics({
            "rmse": rmse,
            "mae": mae,
            "training_time_seconds": training_time
        })
        
        # Save the model artifact based on its specific library requirements
        if "XGBoost" in model_name:
            mlflow.xgboost.log_model(model, artifact_path="model")
        elif "Tree" in model_name or "Forest" in model_name:
            mlflow.sklearn.log_model(
                model, 
                artifact_path="model",
                skops_trusted_types=["sklearn.tree._tree.Tree"]
            )
        else:
            mlflow.sklearn.log_model(model, artifact_path="model")
            
        print(f"Finished {model_name} | RMSE: {rmse:.4f} | MAE: {mae:.4f}\n")

# Register the model

In [ ]:
import mlflow

# 1. Ensure Jupyter is still pointing to DagsHub, not your local server
mlflow.set_tracking_uri("https://dagshub.com/tamer-elkoT/mlops-practitioner.mlflow")

# 2. Use the verified Run ID from your successful Random Forest loop
model_id = "2506ef4a2729474395b3251952f5d987"
model_name = "NYC-prod"
model_uri = f"runs:/{model_id}/model"

# 3. Register the model
registered_model = mlflow.register_model(model_uri=model_uri, name=model_name)
print(f"Success! Registered {model_name} version {registered_model.version}")

In [ ]:
model_name = "Random_Forest"
model_id = "ccef1327b8574ae2bd16f2a8186eeff5"
model_uri = f"runs:/{model_id}/model"

with mlflow.start_run(run_id=model_id):
    mlflow.register_model(model_uri=model_uri, name=model_name)

# Load the Model

In [ ]:
model_version = 1
model_uri = f"models:/{model_name}/{model_version}"

loaded_model = mlflow.pyfunc.load_model(model_uri)


In [ ]:
y_pred = loaded_model.predict(X_test)
y_pred[:5]

# Transition the model to production

In [ ]:
current_modle_uri =f"models:/{model_name}/{model_version}"
production_model_name = "NYC-prod"

client = mlflow.MlflowClient()
client.copy_model_version(src_model_uri=current_modle_uri, dst_name=production_model_name)



In [ ]:
model_version = 1
prod_model_uri = f"models:/{production_model_name}@champion"

loaded_model = mlflow.pyfunc.load_model(prod_model_uri)
y_pred = loaded_model.predict(X_test)
y_pred[:4]

# Export the model to ONNX 

In [ ]:
import onnxruntime as rt
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

In [ ]:
# Define the input shape with a dynamic batch dimension
num_features = X_train_scaled_df.shape[1]


In [ ]:
intial_type = [("float_input"), FloatTensorType([None, num_features])]

# convert the scikit-learn model to ONNX
onnx_model = convert_sklearn()